In [ ]:
!pip install -q transformers datasets sentencepiece accelerate pandas textstat rouge-score bert-score sacrebleu matplotlib

In [ ]:
import gc
import torch
import pandas as pd
import matplotlib.pyplot as plt

from textstat import flesch_kincaid_grade, automated_readability_index
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score

# Model card uses BartTokenizer + BartForConditionalGeneration explicitly
from transformers import BartTokenizer, BartForConditionalGeneration

In [ ]:
# NOTE: facebook/bart-large is a public model — no HuggingFace token required.

# Change this if your file name is different
DATA_PATH = "/content/pilot_20_baseline_eval.csv"

df = pd.read_csv(DATA_PATH)

print("Columns found:")
print(df.columns.tolist())
print("\nNumber of rows:", len(df))

df.head()

In [ ]:
# Required input columns
required_columns = ["source_text", "simple_text"]

missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Input CSV is valid.")

In [ ]:
MODEL_NAME = "eugenesiow/bart-paraphrase"
MODEL_LABEL = "BART-Large (paraphrase fine-tuned)"

OUTPUT_COLUMN = "bart_output"
FKGL_COLUMN = "bart_fkgl"
ARI_COLUMN = "bart_ari"
ROUGE_COLUMN = "bart_rougeL"
BERTSCORE_COLUMN = "bart_bertscore_f1"

In [ ]:
# Create model-specific output columns if they don't already exist
columns_to_create = [
    OUTPUT_COLUMN,
    FKGL_COLUMN,
    ARI_COLUMN,
    ROUGE_COLUMN,
    BERTSCORE_COLUMN
]

for col in columns_to_create:
    if col not in df.columns:
        df[col] = None

print("Model-specific columns are ready.")

In [ ]:
# Model card specifies BartTokenizer and BartForConditionalGeneration.
# Swapped raw facebook/bart-large for eugenesiow/bart-paraphrase: bart-large fine-tuned
# on the Quora, PAWS, and MSR paraphrase corpora. Raw bart-large is a denoising
# autoencoder with no instruction-following ability and little incentive to rewrite
# already-clean text; this checkpoint is actually tuned to rewrite input sentences,
# which is the closest same-architecture fit for "best output this family can give."
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model = BartForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Model loaded.")
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# eugenesiow/bart-paraphrase is paraphrase-tuned, not instruction-tuned -- it still
# cannot follow natural language instructions like the causal LLMs or FLAN-T5.
# Feed the source text directly so it paraphrases/simplifies it, same as raw bart-large.
# max_length=1024 is a hard architectural limit (encoder max_position_embeddings).

def build_prompt(source_text: str) -> str:
    # Feed source text directly -- do NOT add instruction text, it isn't instruction-tuned.
    return source_text.strip()

In [ ]:
def generate_output(source_text: str) -> str:
    prompt = build_prompt(source_text)

    # max_length=1024 is BART's hard architectural limit for the encoder.
    # We truncate here so no tokens are silently dropped with a warning.
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=4,
            early_stopping=True
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text.strip()

In [ ]:
# Test on 2 rows first
for i in range(min(2, len(df))):
    source = str(df.loc[i, "source_text"])
    gold = str(df.loc[i, "simple_text"]) if pd.notna(df.loc[i, "simple_text"]) else ""

    pred = generate_output(source)

    print("ROW:", i)
    print("\nSOURCE:\n", source)
    print("\nGOLD SIMPLE TEXT:\n", gold)
    print("\nMODEL OUTPUT:\n", pred)
    print("\n" + "=" * 100 + "\n")

In [ ]:
# Run the model on all rows
outputs = []

for i, text in enumerate(df["source_text"].fillna("").astype(str).tolist(), start=1):
    pred = generate_output(text)
    outputs.append(pred)
    print(f"Done {i}/{len(df)}")

df[OUTPUT_COLUMN] = outputs
print("Generation complete.")

In [ ]:
def safe_fkgl(text):
    text = str(text).strip()
    if not text:
        return None
    try:
        return float(flesch_kincaid_grade(text))
    except Exception:
        return None

def safe_ari(text):
    text = str(text).strip()
    if not text:
        return None
    try:
        return float(automated_readability_index(text))
    except Exception:
        return None

df[FKGL_COLUMN] = df[OUTPUT_COLUMN].apply(safe_fkgl)
df[ARI_COLUMN] = df[OUTPUT_COLUMN].apply(safe_ari)

df[[OUTPUT_COLUMN, FKGL_COLUMN, ARI_COLUMN]].head()

In [ ]:
# ROUGE-L against human gold simplification
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

rouge_scores = []

for pred, ref in zip(df[OUTPUT_COLUMN].fillna("").astype(str), df["simple_text"].fillna("").astype(str)):
    pred = pred.strip()
    ref = ref.strip()

    if not pred or not ref:
        rouge_scores.append(None)
    else:
        score = scorer.score(ref, pred)
        rouge_scores.append(float(score["rougeL"].fmeasure))

df[ROUGE_COLUMN] = rouge_scores

df[[OUTPUT_COLUMN, "simple_text", ROUGE_COLUMN]].head()

In [ ]:
# BERTScore against human gold simplification
preds = df[OUTPUT_COLUMN].fillna("").astype(str).tolist()
refs = df["simple_text"].fillna("").astype(str).tolist()

valid_indices = []
valid_preds = []
valid_refs = []

for i, (pred, ref) in enumerate(zip(preds, refs)):
    if pred.strip() and ref.strip():
        valid_indices.append(i)
        valid_preds.append(pred)
        valid_refs.append(ref)

bert_f1_scores = [None] * len(df)

if valid_preds:
    P, R, F1 = bertscore_score(valid_preds, valid_refs, lang="en", verbose=False)
    for idx, score in zip(valid_indices, F1):
        bert_f1_scores[idx] = float(score)

df[BERTSCORE_COLUMN] = bert_f1_scores

df[[OUTPUT_COLUMN, "simple_text", BERTSCORE_COLUMN]].head()

In [ ]:
# Create a clean result table without dataset-analysis metadata columns
keep_cols = []

# Keep useful identifiers only if they exist
for col in ["example_id", "act_name", "section_id", "unit_type", "split"]:
    if col in df.columns:
        keep_cols.append(col)

keep_cols += [
    "source_text",
    "simple_text",
    OUTPUT_COLUMN,
    FKGL_COLUMN,
    ARI_COLUMN,
    ROUGE_COLUMN,
    BERTSCORE_COLUMN
]

result_df = df[keep_cols].copy()

print("Clean result table preview:")
result_df.head(10)

In [ ]:
def avg_of_column(dataframe, col_name):
    vals = pd.to_numeric(dataframe[col_name], errors="coerce")
    vals = vals.dropna()
    return None if len(vals) == 0 else round(float(vals.mean()), 3)

summary = {
    "Model": MODEL_LABEL,
    "Rows": len(result_df),
    "Avg FKGL": avg_of_column(result_df, FKGL_COLUMN),
    "Avg ARI": avg_of_column(result_df, ARI_COLUMN),
    "Avg ROUGE-L": avg_of_column(result_df, ROUGE_COLUMN),
    "Avg BERTScore F1": avg_of_column(result_df, BERTSCORE_COLUMN),
}

summary_df = pd.DataFrame([summary])
summary_df

In [ ]:
# Save clean CSV
OUT_PATH = "/content/pilot_20_baseline_eval_with_bart_checked_clean.csv"
result_df.to_csv(OUT_PATH, index=False)

print(f"Saved clean result CSV to: {OUT_PATH}")

In [ ]:
# Create summary table image for screenshot/report
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.axis("off")

plt.title(f"{MODEL_LABEL} - Baseline Evaluation Summary", fontsize=14, pad=14)

table = ax.table(
    cellText=summary_df.values,
    colLabels=summary_df.columns,
    loc="center",
    cellLoc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.6)

IMAGE_PATH = "/content/bart_summary_table.png"
plt.savefig(IMAGE_PATH, bbox_inches="tight", dpi=200)
plt.show()

print(f"Saved summary table image to: {IMAGE_PATH}")

In [ ]:
# Optional: clear memory
del model
del tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memory cleared.")